In [ ]:
import os
import time
import pandas as pd

from google import genai
from google.genai import types
from google.cloud import bigquery
from google.genai.errors import ClientError
from datetime import datetime, timezone


In [ ]:
API_KEY = os.environ["GEMINI_API_KEY"]

client = genai.Client(
    api_key=API_KEY
)

client_bq = bigquery.Client(project="pacey32-agency")

MODEL = "gemini-flash-latest"
PROMPT_VERSION = "1.0"

In [3]:
SYSTEM_PROMPT = """
You are writing relocation guides for professional sports players.

Write a single factual paragraph of approximately 120 words.

Describe:

- where the city is
- what it is known for
- major industries
- culture
- lifestyle
- geography
- anything interesting a newcomer should know

Do not mention sports teams.

Do not mention tax, weather or cost of living.

Do not use bullet points.

Return only the paragraph.
"""

In [ ]:
sql = """
SELECT DISTINCT
    venueLocation,
    state_province,
    country
FROM `pacey32-agency.City.city_tax`
ORDER BY venueLocation
"""

cities = client_bq.query(sql).to_dataframe()

In [ ]:
###
#  One City Test
###

city = "Boston"
state = "Massachusetts"
country = "United States"

prompt = f"""
City: {city}
State/Province: {state}
Country: {country}
"""

response = client.models.generate_content(
    model=MODEL,
    config=types.GenerateContentConfig(
        system_instruction=SYSTEM_PROMPT,
        temperature=0.3,
    ),
    contents=prompt,
)

print(response.text)

Boston, the capital of Massachusetts, is situated on the Atlantic coast in the northeastern United States, anchored by the Charles River and Boston Harbor. Renowned as the historic "Cradle of Liberty," this vibrant city seamlessly blends centuries of rich American heritage with cutting-edge innovation. Its robust economy is driven by global leaders in higher education, biotechnology, healthcare, finance, and technology. Residents experience a sophisticated, highly walkable lifestyle centered around world-class dining, prestigious museums, historic brownstone architecture, and a thriving arts scene. Newcomers will appreciate Boston’s distinct, close-knit neighborhoods—ranging from the charming cobblestone streets of Beacon Hill to the modern energy of the Seaport District—offering an engaging urban atmosphere with effortless access to coastal beaches and nearby mountain escapes.


In [ ]:
summaries = []

for i, row in cities.iterrows():

    prompt = f"""
City: {row.venueLocation}
State/Province: {row.state_province}
Country: {row.country}
"""

    while True:

        try:

            response = client.models.generate_content(
                model="gemini-flash-latest",
                config=types.GenerateContentConfig(
                    system_instruction=SYSTEM_PROMPT,
                    temperature=0.3,
                ),
                contents=prompt,
            )

            summaries.append({
                "venueLocation": row.venueLocation,
                "state_province": row.state_province,
                "country": row.country,
                "summary": response.text.strip(),
                "model": MODEL,
                "prompt_version": PROMPT_VERSION,
                "generated_datetime": datetime.now(timezone.utc)
            })

            print(f"✓ {row.venueLocation}")

            # small gap between requests
            time.sleep(2)

            break

        except ClientError as e:

            if e.code == 429:
                print("Rate limit reached. Sleeping 10 seconds...")
                time.sleep(10)
            else:
                raise

✓ Anaheim
✓ Boston
Rate limit reached. Sleeping 10 seconds...
Rate limit reached. Sleeping 10 seconds...
Rate limit reached. Sleeping 10 seconds...
✓ Buffalo


In [ ]:
summary_df = pd.DataFrame(summaries)

job = client_bq.load_table_from_dataframe(
    summary_df,
    "pacey32-agency.City.city_summary",
    job_config=bigquery.LoadJobConfig(
        write_disposition="WRITE_APPEND"
    )
)

job.result()

print(f"Loaded {len(summary_df)} cities.")

if len(summary_df) == 0:
    print("No new cities to process.")
    exit()

,venueLocation,state_province,country,summary
0,Anaheim,California,United States,Anaheim is a vibrant city located in Southern ...
1,Boston,Massachusetts,United States,Situated along the Atlantic coastline in easte...
2,Buffalo,New York,United States,Situated in Western New York along the eastern...
